In [38]:
import pandas as pd
import numpy as np
import kagglehub
import os
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

## 1. Download dataset and EDA

In [2]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

In [3]:
os.listdir(path)

['IMDB Dataset.csv']

In [4]:
file_path = os.path.join(path, 'IMDB Dataset.csv')
df = pd.read_csv(file_path)

df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
# Imbalances?

df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

This dataset is perfectly balanced.

In [6]:
# Null values?

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


There are not null values.

In [7]:
# Conversion labels:
# Positive -> 1
# Negative -> 0

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

df["label"].value_counts()

label
1    25000
0    25000
Name: count, dtype: int64

In [8]:
# Check length of reviews (number of words)

df["review_length"] = df["review"].apply(lambda x: len(x.split()))
df["review_length"].describe()

count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: review_length, dtype: float64

Since with 300 words I cover more than 75% of the dataset, I consider a max_len = 300 in the following.

In [9]:
# Length of the sequences, it must be fixed

max_len = 300

In [10]:
# Check if there is HTML <br /><br />

print(df["review"].iloc[0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

This must be removed during pre-processing.

## 2. Text pre-processing

In [11]:
# Function to clean text

def clean_text(text):

    text = re.sub(r"<.*?>", " ", text) # Remove HTML
    text = re.sub(r"[^a-zA-Z\s]", "", text) # Remove numbers and special characters
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces

    return text

In [12]:
df["clean_review"] = df["review"].apply(clean_text)

In [13]:
# Check

df["clean_review"].iloc[0]

'one of the other reviewers has mentioned that after watching just oz episode youll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far away i would say the main appeal of the show is due to the fact that it goes where other shows wouldnt dare forget pretty pictu

In [14]:
# Tokenization

df["tokens"] = df["clean_review"].apply(lambda x: x.split())

In [15]:
df

,review,sentiment,label,review_length,clean_review,tokens
0,One of the other reviewers has mentioned that ...,positive,1,307,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,A wonderful little production. <br /><br />The...,positive,1,162,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin..."
2,I thought this was a wonderful way to spend ti...,positive,1,166,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,Basically there's a family where a little boy ...,negative,0,138,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,230,petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon..."
...,...,...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,1,194,i thought this movie did a down right good job...,"[i, thought, this, movie, did, a, down, right,..."
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,0,112,bad plot bad dialogue bad acting idiotic direc...,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,I am a Catholic taught in parochial elementary...,negative,0,230,i am a catholic taught in parochial elementary...,"[i, am, a, catholic, taught, in, parochial, el..."
49998,I'm going to have to disagree with the previou...,negative,0,212,im going to have to disagree with the previous...,"[im, going, to, have, to, disagree, with, the,..."


In [16]:
# Create train, validation and test df

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(len(train_df))
print(len(val_df))
print(len(test_df))

print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())

35000
7500
7500
label
1    17500
0    17500
Name: count, dtype: int64
label
0    3750
1    3750
Name: count, dtype: int64
label
0    3750
1    3750
Name: count, dtype: int64


In [17]:
# Choice of the vocabulary size

all_words = []

for text in train_df["clean_review"]:
    all_words.extend(text.split())

word_counts = Counter(all_words)
total_words = sum(word_counts.values())
unique_words = len(word_counts)

print("Total words: ", total_words)
print("Unique words: ", unique_words)

Total words:  7912492
Unique words:  132532


In [18]:
cumulative = 0

for i, (word, count) in enumerate(word_counts.most_common()):
    cumulative += count
    if cumulative / total_words >= 0.9:
        print("Words needed for 90% coverage: ", i)
        break

Words needed for 90% coverage:  5249


In [19]:
# Vocabulary size

vocab_size = 6000

In [20]:
# Most common words

all_tokens = [token for tokens in df["tokens"] for token in tokens]
counter = Counter(all_tokens)

most_common = counter.most_common(vocab_size)

In [21]:
most_common[:10]

[('the', 664024),
 ('and', 320753),
 ('a', 320610),
 ('of', 288490),
 ('to', 266937),
 ('is', 210520),
 ('in', 185078),
 ('it', 154929),
 ('i', 152153),
 ('this', 149912)]

I consider the index:
* 0 for \<PAD>
* 1 for \<UNK>

In [22]:
word2idx = {"<PAD>": 0, "<UNK>": 1}

for idx, (word, _) in enumerate(most_common, start=2):
    word2idx[word] = idx

idx2word = {idx: word for word, idx in word2idx.items()}

In [23]:
# Function for encoding and padding

def encode_and_padding(tokens):

    tokens = tokens[-max_len:] # Truncate from the beginning
    length = len(tokens)
    encoded = [word2idx.get(token, word2idx["<UNK>"]) for token in tokens] # Convert to indices

    if length < max_len: # Padding
        encoded += [word2idx["<PAD>"]] * (max_len - length)

    return encoded, length

In [24]:
for dataset in [train_df, test_df, val_df]:
    dataset["encoded"], dataset["length"] = zip(*dataset["tokens"].apply(lambda x: encode_and_padding(x)))

In [25]:
test_df

,review,sentiment,label,review_length,clean_review,tokens,encoded,length
42715,My parents used to rent a lot of horror movies...,negative,0,194,my parents used to rent a lot of horror movies...,"[my, parents, used, to, rent, a, lot, of, horr...","[53, 739, 325, 6, 822, 4, 165, 5, 192, 92, 50,...",194
15031,I got a chance to see this movie at an early s...,positive,1,148,i got a chance to see this movie at an early s...,"[i, got, a, chance, to, see, this, movie, at, ...","[10, 186, 4, 558, 6, 64, 11, 17, 30, 33, 402, ...",144
9762,Well the previews looked funny and I usually d...,negative,0,167,well the previews looked funny and i usually d...,"[well, the, previews, looked, funny, and, i, u...","[70, 2, 5391, 567, 153, 3, 10, 610, 89, 138, 6...",163
19036,As a true lover of film I must advise you to a...,negative,0,124,as a true lover of film i must advise you to a...,"[as, a, true, lover, of, film, i, must, advise...","[14, 4, 285, 1531, 5, 19, 10, 215, 4148, 22, 6...",119
32649,I first saw this movie as a teenager when it c...,positive,1,128,i first saw this movie as a teenager when it c...,"[i, first, saw, this, movie, as, a, teenager, ...","[10, 88, 205, 11, 17, 14, 4, 2119, 50, 9, 368,...",121
...,...,...,...,...,...,...,...,...
18842,"Believe it or not, the Mona Lisa actually got ...",positive,1,530,believe it or not the mona lisa actually got s...,"[believe, it, or, not, the, mona, lisa, actual...","[5, 1, 1380, 1, 45, 4, 1, 16, 3893, 1, 61, 1, ...",300
5957,"There is no way to describe how really, really...",negative,0,248,there is no way to describe how really really ...,"[there, is, no, way, to, describe, how, really...","[47, 7, 55, 95, 6, 1618, 85, 62, 62, 62, 78, 1...",246
32589,"I felt that the film was rushed, and the actin...",negative,0,80,i felt that the film was rushed and the acting...,"[i, felt, that, the, film, was, rushed, and, t...","[10, 424, 12, 2, 19, 13, 3247, 3, 2, 111, 13, ...",78
12063,Pretty dreadful movie about several unbalanced...,negative,0,52,pretty dreadful movie about several unbalanced...,"[pretty, dreadful, movie, about, several, unba...","[180, 1997, 17, 42, 430, 1, 185, 83, 8, 4, 510...",52


In [26]:
# Check

print(len(test_df["clean_review"].iloc[0]))
print(len(test_df["tokens"].iloc[0]))
print(len(test_df["encoded"].iloc[0]))

963
194
300


## 3. GRU RNN

In [27]:
# Modification of the dataset

class IMDBDataset(Dataset):

    def __init__(self, dataframe):
        self.sequences = dataframe["encoded"].values
        self.labels = dataframe["label"].values
        self.lengths = dataframe["length"].values

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = torch.tensor(self.sequences[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float)
        l = torch.tensor(self.lengths[idx], dtype=torch.long)
        return x, y, l

In [28]:
# Create datasets torch

batch_size = 64

train_dataset = IMDBDataset(train_df)
val_dataset = IMDBDataset(val_df)
test_dataset = IMDBDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [29]:
# Model

class GRUModel(nn.Module):
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(GRUModel, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

        # GRU layer
        self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

        # FC layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):

        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.gru(packed)
        last_hidden = hidden[-1]
        logits = self.fc(last_hidden)

        return logits.squeeze(1)

In [30]:
# Hyperparameters

vocab_size = len(word2idx)
embedding_dim = 100
hidden_dim = 128
num_layers = 1

In [31]:
# Setup training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GRUModel(vocab_size=vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, num_layers=num_layers).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [32]:
# Training function

def train_epoch(model, loader, criterion, optimizer, device):

    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y, lengths in loader:

        x = x.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()   
        
        logits = model(x, lengths)
        loss = criterion(logits, y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        preds = torch.sigmoid(logits) >= 0.5
        correct += (preds.float() == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total

In [33]:
# Function to evaluate

def evaluate(model, loader, criterion, device):

    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y, lengths in loader:

            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            logits = model(x, lengths)
            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.sigmoid(logits) >= 0.5
            correct += (preds.float() == y).sum().item()
            total += y.size(0)

        return total_loss / len(loader), correct / total

In [34]:
# Complete loop

num_epochs = 10
patience = 3

best_val_loss = float("inf")
counter= 0

for epoch in range(num_epochs):

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_gru_model.pt")
    else:
        counter += 1
        print(f"No improvement, patience counter {counter} / {patience}")

        if counter >= patience:
            print("Early stopping")
            break

Epoch 1/10
Train Loss: 0.6229 | Train Acc: 0.6448
Val   Loss: 0.5116 | Val   Acc: 0.7617
----------------------------------------
Epoch 2/10
Train Loss: 0.3779 | Train Acc: 0.8342
Val   Loss: 0.3215 | Val   Acc: 0.8641
----------------------------------------
Epoch 3/10
Train Loss: 0.2635 | Train Acc: 0.8923
Val   Loss: 0.2880 | Val   Acc: 0.8797
----------------------------------------
Epoch 4/10
Train Loss: 0.2126 | Train Acc: 0.9159
Val   Loss: 0.2673 | Val   Acc: 0.8929
----------------------------------------
Epoch 5/10
Train Loss: 0.1737 | Train Acc: 0.9341
Val   Loss: 0.2995 | Val   Acc: 0.8787
----------------------------------------
No improvement, patience counter 1 / 3
Epoch 6/10
Train Loss: 0.1341 | Train Acc: 0.9517
Val   Loss: 0.3191 | Val   Acc: 0.8869
----------------------------------------
No improvement, patience counter 2 / 3
Epoch 7/10
Train Loss: 0.0997 | Train Acc: 0.9665
Val   Loss: 0.3237 | Val   Acc: 0.8889
----------------------------------------
No improveme

In [35]:
# Upload best model

model.load_state_dict(torch.load("best_gru_model.pt"))

C:\Users\Chiara\AppData\Local\Temp\ipykernel_15392\3705041530.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_gru_model.pt"))


<All keys matched successfully>

In [37]:
# Evaluation on the test set

test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.2606
Test Accuracy: 0.8955


In [39]:
# Confusion matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for x, y, lengths in test_loader:
        x = x.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[3391  359]
 [ 425 3325]]


## 4. LSTM RNN

In [40]:
# Model

class LSTMModel(nn.Module):
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(LSTMModel, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

        # LSTM layer
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

        # FC layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):

        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed)
        last_hidden = hidden[-1]
        logits = self.fc(last_hidden)

        return logits.squeeze(1)

In [41]:
# Setup training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(vocab_size=vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, num_layers=num_layers).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [42]:
# Complete loop

num_epochs = 10
patience = 3

best_val_loss = float("inf")
counter= 0

for epoch in range(num_epochs):

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_lstm_model.pt")
    else:
        counter += 1
        print(f"No improvement, patience counter {counter} / {patience}")

        if counter >= patience:
            print("Early stopping")
            break

Epoch 1/10
Train Loss: 0.6233 | Train Acc: 0.6499
Val   Loss: 0.5514 | Val   Acc: 0.7292
----------------------------------------
Epoch 2/10
Train Loss: 0.5542 | Train Acc: 0.7257
Val   Loss: 0.7512 | Val   Acc: 0.7171
----------------------------------------
No improvement, patience counter 1 / 3
Epoch 3/10
Train Loss: 0.5269 | Train Acc: 0.7375
Val   Loss: 0.4737 | Val   Acc: 0.7829
----------------------------------------
Epoch 4/10
Train Loss: 0.4033 | Train Acc: 0.8256
Val   Loss: 0.4878 | Val   Acc: 0.7607
----------------------------------------
No improvement, patience counter 1 / 3
Epoch 5/10
Train Loss: 0.3650 | Train Acc: 0.8425
Val   Loss: 0.3554 | Val   Acc: 0.8392
----------------------------------------
Epoch 6/10
Train Loss: 0.3023 | Train Acc: 0.8758
Val   Loss: 0.3355 | Val   Acc: 0.8576
----------------------------------------
Epoch 7/10
Train Loss: 0.2645 | Train Acc: 0.8915
Val   Loss: 0.3216 | Val   Acc: 0.8657
----------------------------------------
Epoch 8/10
T

In [43]:
# Upload best model

model.load_state_dict(torch.load("best_lstm_model.pt"))

C:\Users\Chiara\AppData\Local\Temp\ipykernel_15392\1627835130.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_lstm_model.pt"))


<All keys matched successfully>

In [44]:
# Evaluation on the test set

test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.2933
Test Accuracy: 0.8817


In [45]:
# Confusion matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for x, y, lengths in test_loader:
        x = x.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[3270  480]
 [ 407 3343]]


I conclude that GRU is performing better than LSTM (accuracy: 89.55% vs 88.17%). This can be due to 2 main things:
* The dataset is not too big.
* Sentences are quite short.